# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. All entities are referenced by their `@id` as defined in the dataset Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata attributes
print("Dataset Title: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)
print("Version: ", dataset.metadata.version)
print("Published date: ", dataset.metadata.datePublished)
print("Citation: ", dataset.metadata.citeAs)
print("License: ", dataset.metadata.license)
print("Keywords: ", ', '.join(getattr(dataset.metadata, 'keywords', [])))

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` values for reproducibility.


In [ ]:
# List all record sets defined in the schema (by `@id`)
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No top-level record sets found in metadata. Attempting to discover from distributions...")
    # Sometimes, record sets may be defined as distribution or via hasPart.
    # For this dataset, we'll attempt to fetch from distribution
    dists = getattr(dataset.metadata, "distribution", [])
    for dist in dists:
        print(f"Distribution: {getattr(dist, '@id', dist)}")
    # As a fallback, mlcroissant will usually provide a default record set
    print("Using the default record set inferred by mlcroissant (usually the first tabular file in distribution).\n")
    # Attempt to retrieve all records by leaving record_set=None
    sample_records = list(dataset.records(record_set=None))
    print("Example records (first 2 rows):")
    for rec in sample_records[:2]:
        print(rec)
else:
    print("Record Sets (by @id):")
    for r in record_sets:
        print(" -", getattr(r, '@id', r))
    # Optionally, list fields of each record set
    first_rs = record_sets[0]
    print(f"\nFirst Record Set `{getattr(first_rs, '@id', first_rs)}` fields and columns:")
    for field in getattr(first_rs, "field", []):
        print(f"  Field @id: {getattr(field, '@id', field)} | label: {getattr(field, 'name', field)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Since the dataset defines no explicit recordSet, records are loaded from the default tabular file.
print("Loading all records from main record set...")
all_records = list(dataset.records(record_set=None))

df = pd.DataFrame(all_records)
print("DataFrame shape:", df.shape)
print("Columns (field @id):\n", list(df.columns))
# Show sample records
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by their `@id` as available in the DataFrame.


In [ ]:
# List numeric fields (by column '@id')
import numpy as np
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns (by @id):", numeric_fields)

# For demonstration, pick 'cr:field:Coverage(%)', if present, to filter and normalize.
# Otherwise, try 'cr:field:MW' or any available numeric field.
if 'cr:field:Coverage(%)' in df.columns:
    numeric_field_id = 'cr:field:Coverage(%)'
elif 'cr:field:MW' in df.columns:
    numeric_field_id = 'cr:field:MW'
elif numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError("No numeric field found in the DataFrame.")

# Apply filtering & normalization
threshold = df[numeric_field_id].mean()  # Use mean as threshold for demonstration
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (filtered {len(filtered_df)} records):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
field_norm = f"{numeric_field_id}_normalized"
filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, field_norm]].head())

# Attempt a groupby by protein description if present
group_field_id = None
if 'cr:field:Description' in df.columns:
    group_field_id = 'cr:field:Description'
elif 'cr:field:Gene Name' in df.columns:
    group_field_id = 'cr:field:Gene Name'
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All references are made using the field `@id`s as per the schema.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
df[numeric_field_id].hist(bins=30)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field_id}')
plt.grid(True)
plt.show()

# If there are two numeric fields, plot scatter plot
if len(numeric_fields) >= 2:
    plt.figure(figsize=(6, 6))
    plt.scatter(df[numeric_fields[0]], df[numeric_fields[1]], alpha=0.5)
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.title(f'Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]}')
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all elements by their `@id`. We performed an overview, analyzed numeric distributions, filtered and normalized records, and visualized data fields. This workflow can be adapted to other Croissant-structured datasets for transparent and reproducible data science.